In [1]:
# Getting imports and the model builder
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
sys.path.insert(0, "../baseline")

from smolagents import OpenAIModel
from dotenv import load_dotenv
load_dotenv()

MODEL_NAMES = ["Qwen/Qwen3.5-9B"]
WINDOW_SIZES = [5]


def build_model(model_name: str) -> OpenAIModel:
    if model_name in ["gpt-4o", "gpt-5.4-mini"]:
        return OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served
    # open-weight models -- verified this works for both Qwen3.7-Plus and Qwen3.5-9B.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within
    # 5 minutes instead of hanging indefinitely -- evaluate_agent already catches such errors.
    return OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        client_kwargs={"timeout": 300.0},
    )

In [2]:
# Getting the tools setup and the MarkovReActCodeAgent builder
from smolagents.monitoring import LogLevel
from common_setup import assert_no_reasoning, build_tools
from smolagents.markov_react import MarkovReActCodeAgent


def build_agent(model, model_name: str, window_size: int, tools):
    return MarkovReActCodeAgent(
        tools=tools,
        model=model,
        max_steps=50,
        verbosity_level=LogLevel.DEBUG,
        additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
        window_size=window_size,
        stream_outputs="Qwen" in model_name,
        step_callbacks=[assert_no_reasoning] if "Qwen" in model_name else None,
    )

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Fermi (RealFP) dataset

Same RealFP split, `fermi_scorer`, and `FERMI_ANSWER_INSTRUCTION` used in
[naiveReAct_fermi.ipynb](../baseline/naiveReAct_fermi.ipynb), reused here so MarkovReAct stays
directly comparable to naive ReAct on Fermi -- same dataset, same scorer, same prompt suffix.

In [3]:
# Load the Fermi RealFP validation split
import pandas as pd
from common_setup import evaluate_agent
from fermi_setup import load_fermi_dataset, fermi_scorer, FERMI_ANSWER_INSTRUCTION

eval_ds = load_fermi_dataset(split="val")
print(f"Loaded {len(eval_ds)} examples")

Loaded 125 examples


In [4]:
# One model (and its tools) is built once, then reused across all 3 window sizes.
all_results = {}

for model_name in MODEL_NAMES:
    model = build_model(model_name)
    tools, ti_tool, visualizer = build_tools(model)

    for window_size in WINDOW_SIZES:
        agent = build_agent(model, model_name, window_size, tools)
        safe_model_name = model_name.replace("/", "_")
        pickle_dir = f"markov_fermi_{safe_model_name}_w{window_size}"
        print(f"\n=== {model_name}, window_size={window_size} ===")
        results = evaluate_agent(
            agent,
            eval_ds,
            ti_tool,
            visualizer,
            output_file=f"{pickle_dir}.jsonl",
            pickle_dir=pickle_dir,
            scorer=fermi_scorer,
            question_suffix=FERMI_ANSWER_INSTRUCTION,
        )
        all_results[(model_name, window_size)] = results


=== Qwen/Qwen3.5-9B, window_size=5 ===

[1/125] If all but 1 million people on Earth died, how far (on average) would you have to walk to meet someo...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If all but 1 million people on Earth died, how far (on average) would you have to walk to meet someone?         │
│                                                                                                                 │
│ Give your final answer as a number with units matching the question (e.g. "22.3 km", "79 g"; use a bare number  │
│ like "33000" only if the question has no natural unit). Do not include filler words like "about" or "roughly",  │
│ or thousands separators. Write the full number of digits (e.g. "256000000000") rather than word multipliers     │
│ like "million" or "billion".                                                                                    │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3.5-9B ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output()

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import math                                                                                                      
                                                                                                                   
  # Earth's surface area (in square meters)                                                                        
  # Earth's radius is about 6,371 km, surface area = 4 * pi * r^2                                                  
  earth_radius_km = 6371                                                                                           
  earth_surface_area_sq_m = 4 * math.pi * (earth_radius_km * 1000)**2                                              
  print(f"Earth surface area: {earth_surface_area_sq_m:.2e} square meters")                                        
                                                                                                                   
  # Number of people remaining                                                                                     
  people_remaining = 1_000_000                                                                                     
  print(f"People remaining: {people_remaining}")                                                                   
                                                                                                                   
  # Average area per person (m²)                                                                                   
  area_per_person_sq_m = earth_surface_area_sq_m / people_remaining                                                
  print(f"Area per person: {area_per_person_sq_m:.2e} square meters")                                              
                                                                                                                   
  # Expected distance to nearest neighbor                                                                          
  # For 2D Poisson process: E[d] = 0.75 * sqrt(A/(N)) where A is total area, N is number of points                 
  # More precisely: for average nearest neighbor distance in 2D uniform distribution:                              
  # E[d] = 1 / (2 * sqrt(n/area)) where n = N is number of points                                                  
  n_per_area = people_remaining / earth_surface_area_sq_m  # density in people per m²                              
  avg_distance_m = 1 / (2 * math.sqrt(n_per_area))                                                                 
  print(f"Density: {n_per_area:.10e} people per m²")                                                               
  print(f"Average nearest neighbor distance (theoretical, 2D Poisson): {avg_distance_m:.4f} meters")               
                                                                                                                   
  # Let's also calculate using the √(A/(πN)) approximation                                                         
  avg_distance_alt = math.sqrt(earth_surface_area_sq_m / (math.pi * people_remaining))                             
  print(f"Alternative calculation (sqrt(A/(πN))): {avg_distance_alt:.4f} meters")                                  
                                                                                                                   
  # Convert to kilometers                                                                                          
  avg_distance_km = avg_distance_m / 1000                                                                          
  print(f"\nAverage distance in meters: {avg_distance_m}")                                                         
  print(f"Average distance in kilometers: {avg_distance_km}")                                                      
 ────────────────────────────────────────────────────────

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_value = round(1103.76)                                                                                     
  print(final_value, "metres")                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
1104 metres

Out: None

[Step 4: Duration 1.76 seconds| Input tokens: 23,405 | Output tokens: 1,195]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output()

In [5]:
rows = []
for (model_name, window_size), results in all_results.items():
    df = pd.DataFrame(results)
    total = len(df)
    avg_score = df["is_correct"].mean()  # continuous 0-1 order-of-magnitude accuracy metric, not a boolean fraction
    total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0))
    rows.append({
        "Model": model_name,
        "window_size": window_size,
        "Avg Score": round(avg_score, 3),
        "Avg Steps": round(df["num_steps"].mean(), 1),
        "Avg Tokens": round(total_tokens.mean()),
    })

pd.DataFrame(rows)

,Model,window_size,Avg Score,Avg Steps,Avg Tokens
0,Qwen/Qwen3.5-9B,5,0.33,9.1,74951
